In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from joblib import load

ROOT = os.path.abspath("..")

# Load model and optional scaler
model = load(os.path.join(ROOT, 'data', 'models', 'gbr_model.pkl'))
scaler_path = os.path.join(ROOT, 'data', 'models', 'scaler.pkl')
scaler = None
if os.path.exists(scaler_path):
    scaler = load(scaler_path)

# Load dataset (Excel or CSV)
possible = [os.path.join(ROOT, 'data', 'raw', 'dispersion_vs_L.xlsx'),
            os.path.join(ROOT, 'data', 'dispersion_vs_L.xlsx'),
            os.path.join(ROOT, 'data', 'raw', 'dispersion_vs_L.csv'),
            os.path.join(ROOT, 'data', 'dispersion_vs_L.csv')]
data_path = next((p for p in possible if os.path.exists(p)), None)
if data_path is None:
    raise FileNotFoundError('dispersion_vs_L data not found')
if data_path.endswith('.csv'):
    data = pd.read_csv(data_path)
else:
    data = pd.read_excel(data_path)

X = data[["kL","L"]].values
y = data["c_beta"].values

# Scale if scaler available
X_in = scaler.transform(X) if scaler is not None else X

# Predict and plot
pred = model.predict(X_in)

plt.figure(figsize=(7,6))
plt.scatter(y, pred, s=10, alpha=0.6)
plt.xlabel('Analytical c')
plt.ylabel('ML predicted c')
plt.title('ML Surrogate Accuracy (all data)')
plt.grid(True)
plt.tight_layout()
plot_dir = os.path.join(ROOT, 'results', 'plots')
os.makedirs(plot_dir, exist_ok=True)
plot_path = os.path.join(plot_dir, 'predictions_vs_actual.png')
plt.savefig(plot_path, dpi=300)
plt.show()
print(f'Plot saved at: {plot_path}')